# Previous Application Data Cleaning

This notebook cleans the previous-application records using the problems found during EDA. It keeps records linked to the project applicants, fixes the placeholder date values, groups rare categories, flags unusual amounts, and removes features with too many missing values.


## Import libraries


In [3]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 150)
print("Libraries imported successfully.")

Libraries imported successfully.


## Set project paths


In [5]:
current_folder = Path.cwd().resolve()
project_root = current_folder.parent if current_folder.name == "notebooks" else current_folder
raw_path = project_root / "data" / "raw" / "previous_application.csv"
training_ids_path = project_root / "data" / "modeling" / "splits" / "training_ids.csv"
test_ids_path = project_root / "data" / "modeling" / "splits" / "test_ids.csv"
output_path = project_root / "data" / "interim" / "previous_application_clean.pkl"
audit_folder = project_root / "reports" / "audits"
output_path.parent.mkdir(parents=True, exist_ok=True)
audit_folder.mkdir(parents=True, exist_ok=True)
for required_path in [raw_path, training_ids_path, test_ids_path]:
    assert required_path.exists(), f"Required file was not found: {required_path}"
print("Raw input:", raw_path)
print("Clean output:", output_path)

Raw input: /Users/taranveersingh/A-MRP/data/raw/previous_application.csv
Clean output: /Users/taranveersingh/A-MRP/data/interim/previous_application_clean.pkl


## Load the data and retain project applicants


In [7]:
previous_raw = pd.read_csv(raw_path)
training_ids = pd.read_csv(training_ids_path)["SK_ID_CURR"]
test_ids = pd.read_csv(test_ids_path)["SK_ID_CURR"]
training_id_set = set(training_ids)
project_ids = training_id_set.union(set(test_ids))
original_rows, original_columns = previous_raw.shape

previous_clean = previous_raw.loc[previous_raw["SK_ID_CURR"].isin(project_ids)].copy().reset_index(drop=True)
out_of_scope_rows = original_rows - len(previous_clean)
print("Raw rows:", original_rows)
print("Project rows retained:", len(previous_clean))
print("Out-of-scope rows removed:", out_of_scope_rows)
print("Applicants with previous applications:", previous_clean["SK_ID_CURR"].nunique())

Raw rows: 1670214
Project rows retained: 1413701
Out-of-scope rows removed: 256513
Applicants with previous applications: 291057


The rows that were removed belong to applicants outside the labelled training and test data, so they are not needed here.


## Validate identifiers and remove exact duplicates


In [10]:
missing_current_ids = int(previous_clean["SK_ID_CURR"].isna().sum())
missing_previous_ids = int(previous_clean["SK_ID_PREV"].isna().sum())
duplicate_previous_ids = int(previous_clean.duplicated("SK_ID_PREV").sum())
exact_duplicates = int(previous_clean.duplicated().sum())
if exact_duplicates > 0:
    previous_clean = previous_clean.drop_duplicates().reset_index(drop=True)
assert missing_current_ids == 0 and missing_previous_ids == 0
assert previous_clean["SK_ID_PREV"].is_unique, "SK_ID_PREV must be unique."
print("Missing applicant IDs:", missing_current_ids)
print("Missing previous-application IDs:", missing_previous_ids)
print("Duplicate previous IDs before cleaning:", duplicate_previous_ids)
print("Exact duplicate rows removed:", exact_duplicates)

Missing applicant IDs: 0
Missing previous-application IDs: 0
Duplicate previous IDs before cleaning: 0
Exact duplicate rows removed: 0


No missing or duplicate IDs were found, so nothing needed to be removed here.


## Correct Home Credit placeholder values


In [13]:
date_columns = [
    "DAYS_FIRST_DRAWING", "DAYS_FIRST_DUE", "DAYS_LAST_DUE_1ST_VERSION",
    "DAYS_LAST_DUE", "DAYS_TERMINATION"
]
placeholder_counts = {}
for column in date_columns:
    placeholder_counts[column] = int(previous_clean[column].eq(365243).sum())
    previous_clean[column] = previous_clean[column].replace(365243, np.nan)
placeholder_total = sum(placeholder_counts.values())
print("Date placeholders converted to missing:", placeholder_total)
pd.Series(placeholder_counts, name="placeholder_count")

Date placeholders converted to missing: 1286947


DAYS_FIRST_DRAWING           799094
DAYS_FIRST_DUE                33962
DAYS_LAST_DUE_1ST_VERSION     79099
DAYS_LAST_DUE                180792
DAYS_TERMINATION             194000
Name: placeholder_count, dtype: int64

365243 is not a real date, so it is converted to missing across all five date columns. DAYS_FIRST_DRAWING has the most placeholders, since a lot of previous applications never had funds drawn.


## Standardize categorical values


In [16]:
categorical_columns = previous_clean.select_dtypes(exclude="number").columns.tolist()
for column in categorical_columns:
    previous_clean[column] = previous_clean[column].str.strip().fillna("Unknown")

training_mask = previous_clean["SK_ID_CURR"].isin(training_id_set)
rare_category_changes = 0
for column in categorical_columns:
    counts = previous_clean.loc[training_mask, column].value_counts()
    rare_values = counts[counts < 100].index
    change_mask = previous_clean[column].isin(rare_values)
    rare_category_changes += int(change_mask.sum())
    previous_clean.loc[change_mask, column] = "Other rare"
print("Categorical columns standardized:", len(categorical_columns))
print("Rare-category cells consolidated:", rare_category_changes)

Categorical columns standardized: 16
Rare-category cells consolidated: 518


Text values are cleaned up, and rare categories (fewer than 100 training records) are grouped into an Other rare bucket, so categories with very little data do not cause problems later.


## Audit invalid and unusual numerical values


In [19]:
future_decision = previous_clean["DAYS_DECISION"].gt(0)
seller_area_placeholder = previous_clean["SELLERPLACE_AREA"].eq(-1)
credit_above_application = previous_clean["AMT_CREDIT"].gt(previous_clean["AMT_APPLICATION"])
negative_down_payment = previous_clean["AMT_DOWN_PAYMENT"].lt(0)
zero_application = previous_clean["AMT_APPLICATION"].eq(0)

previous_clean["PREV_DATE_ANOMALY"] = future_decision.astype("int8")
previous_clean["PREV_SELLER_AREA_MISSING"] = seller_area_placeholder.astype("int8")
previous_clean["PREV_CREDIT_ABOVE_APPLICATION"] = credit_above_application.fillna(False).astype("int8")
previous_clean["PREV_NEGATIVE_DOWN_PAYMENT"] = negative_down_payment.fillna(False).astype("int8")
previous_clean["PREV_ZERO_APPLICATION"] = zero_application.fillna(False).astype("int8")
previous_clean.loc[future_decision, "DAYS_DECISION"] = np.nan
previous_clean.loc[seller_area_placeholder, "SELLERPLACE_AREA"] = np.nan
print("Future decision dates corrected:", int(future_decision.sum()))
print("Seller-area placeholders corrected:", int(seller_area_placeholder.sum()))
print("Credit above requested amount records:", int(credit_above_application.sum()))
print("Negative down-payment records retained and flagged:", int(negative_down_payment.sum()))
print("Zero requested-amount records flagged:", int(zero_application.sum()))

Future decision dates corrected: 0
Seller-area placeholders corrected: 636583
Credit above requested amount records: 560467
Negative down-payment records retained and flagged: 2
Zero requested-amount records flagged: 325302


No future decision dates were found. A large number of seller-area values are placeholders (-1), so they are converted to missing. Credit above the requested amount and zero requested amounts are common, so these are kept and flagged rather than removed, since they may be genuine.


## Build training-only feature decisions


In [22]:
MISSING_THRESHOLD = 0.50
training_previous = previous_clean.loc[previous_clean["SK_ID_CURR"].isin(training_id_set)]
decision_rows = []
for column in previous_clean.columns:
    if column in ["SK_ID_CURR", "SK_ID_PREV"]:
        continue
    series = training_previous[column]
    missing_rate = series.isna().mean()
    unique_non_missing = series.nunique(dropna=True)
    decision = "Keep"
    reason = "Retain for applicant-level aggregation and later target-based selection"
    if missing_rate >= MISSING_THRESHOLD:
        decision = "Remove"
        reason = f"Training-linked missing rate is at least {MISSING_THRESHOLD:.0%}"
    elif unique_non_missing <= 1:
        decision = "Remove"
        reason = "Constant among training-linked records"
    decision_rows.append({
        "feature": column, "data_type": str(series.dtype),
        "missing_count": int(series.isna().sum()), "missing_rate": missing_rate,
        "unique_non_missing": int(unique_non_missing), "decision": decision,
        "reason": reason, "target_association_stage": "After applicant-level aggregation"
    })
feature_decisions = pd.DataFrame(decision_rows).sort_values(
    ["decision", "missing_rate"], ascending=[True, False]
).reset_index(drop=True)
removed_features = feature_decisions.loc[feature_decisions["decision"] == "Remove", "feature"].tolist()
previous_clean = previous_clean.drop(columns=removed_features)
print("Features removed:", removed_features)
feature_decisions.round(5)

Features removed: ['RATE_INTEREST_PRIMARY', 'RATE_INTEREST_PRIVILEGED', 'DAYS_FIRST_DRAWING', 'DAYS_TERMINATION', 'AMT_DOWN_PAYMENT', 'RATE_DOWN_PAYMENT', 'DAYS_LAST_DUE', 'PREV_DATE_ANOMALY']


,feature,data_type,missing_count,missing_rate,unique_non_missing,decision,reason,target_association_stage
0,DAYS_LAST_DUE_1ST_VERSION,float64,512327,0.45301,4594,Keep,Retain for applicant-level aggregation and lat...,After applicant-level aggregation
1,SELLERPLACE_AREA,float64,509538,0.45054,2030,Keep,Retain for applicant-level aggregation and lat...,After applicant-level aggregation
2,DAYS_FIRST_DUE,float64,476293,0.42115,2891,Keep,Retain for applicant-level aggregation and lat...,After applicant-level aggregation
3,NFLAG_INSURED_ON_APPROVAL,float64,449078,0.39708,2,Keep,Retain for applicant-level aggregation and lat...,After applicant-level aggregation
4,AMT_GOODS_PRICE,float64,255678,0.22608,77785,Keep,Retain for applicant-level aggregation and lat...,After applicant-level aggregation
5,AMT_ANNUITY,float64,245806,0.21735,293006,Keep,Retain for applicant-level aggregation and lat...,After applicant-level aggregation
6,CNT_PAYMENT,float64,245803,0.21734,47,Keep,Retain for applicant-level aggregation and lat...,After applicant-level aggregation
7,AMT_CREDIT,float64,1,0.00000,75823,Keep,Retain for applicant-level aggregation and lat...,After applicant-level aggregation
8,NAME_CONTRACT_TYPE,str,0,0.00000,4,Keep,Retain for applicant-level aggregation and lat...,After applicant-level aggregation
9,AMT_APPLICATION,float64,0,0.00000,77785,Keep,Retain for applicant-level aggregation and lat...,After applicant-level aggregation


8 features were removed, mostly because they were missing more than half their values in the training data. This includes the two interest-rate columns and DAYS_FIRST_DRAWING, the same features flagged as very high-missing during EDA. PREV_DATE_ANOMALY was also removed, since no future decision dates were actually found, so it ended up being constant.


## Fill categorical missingness and record remaining missingness


In [25]:
retained_categorical = previous_clean.select_dtypes(exclude="number").columns.tolist()
categorical_missing_before = int(previous_clean[retained_categorical].isna().sum().sum())
previous_clean[retained_categorical] = previous_clean[retained_categorical].fillna("Unknown")
record_features = [c for c in previous_clean.columns if c not in ["SK_ID_CURR", "SK_ID_PREV"]]
previous_clean["PREV_RECORD_MISSING_COUNT"] = previous_clean[record_features].isna().sum(axis=1).astype("int8")
previous_clean["PREV_RECORD_MISSING_RATE"] = previous_clean["PREV_RECORD_MISSING_COUNT"] / len(record_features)
print("Categorical missing values filled:", categorical_missing_before)
print(previous_clean[["PREV_RECORD_MISSING_COUNT", "PREV_RECORD_MISSING_RATE"]].describe().round(4))

Categorical missing values filled: 0
       PREV_RECORD_MISSING_COUNT  PREV_RECORD_MISSING_RATE
count               1.413701e+06              1.413701e+06
mean                2.381600e+00              7.440000e-02
std                 2.664600e+00              8.330000e-02
min                 0.000000e+00              0.000000e+00
25%                 0.000000e+00              0.000000e+00
50%                 1.000000e+00              3.120000e-02
75%                 4.000000e+00              1.250000e-01
max                 7.000000e+00              2.188000e-01


These two columns track how much information is missing for each previous-application record. On average, a record is missing about 7% of its fields.


## Validate the cleaned table


In [28]:
numeric_clean = previous_clean.select_dtypes(include="number")
infinite_count = int(np.isinf(numeric_clean.to_numpy()).sum())
remaining_placeholder_dates = int(previous_clean[[c for c in date_columns if c in previous_clean.columns]].eq(365243).sum().sum())
validation_checks = pd.DataFrame([
    {"check": "Only project applicants retained", "passed": set(previous_clean["SK_ID_CURR"]).issubset(project_ids)},
    {"check": "Applicant IDs complete", "passed": previous_clean["SK_ID_CURR"].notna().all()},
    {"check": "Previous-application IDs complete", "passed": previous_clean["SK_ID_PREV"].notna().all()},
    {"check": "Previous-application IDs unique", "passed": previous_clean["SK_ID_PREV"].is_unique},
    {"check": "No exact duplicates", "passed": not previous_clean.duplicated().any()},
    {"check": "No categorical missing values", "passed": previous_clean.select_dtypes(exclude="number").isna().sum().sum() == 0},
    {"check": "No 365243 date placeholders", "passed": remaining_placeholder_dates == 0},
    {"check": "No future decision dates", "passed": not previous_clean["DAYS_DECISION"].gt(0).any()},
    {"check": "Interest-rate columns removed by missingness rule", "passed": "RATE_INTEREST_PRIMARY" not in previous_clean.columns and "RATE_INTEREST_PRIVILEGED" not in previous_clean.columns},
    {"check": "No infinite numerical values", "passed": infinite_count == 0},
])
assert validation_checks["passed"].all(), "At least one previous-application cleaning check failed."
validation_checks

,check,passed
0,Only project applicants retained,True
1,Applicant IDs complete,True
2,Previous-application IDs complete,True
3,Previous-application IDs unique,True
4,No exact duplicates,True
5,No categorical missing values,True
6,No 365243 date placeholders,True
7,No future decision dates,True
8,Interest-rate columns removed by missingness rule,True
9,No infinite numerical values,True


All checks passed.


## Save the clean table and audit reports


In [31]:
cleaning_audit = pd.DataFrame([
    {"rule": "Out-of-scope records removed", "affected": out_of_scope_rows},
    {"rule": "Exact duplicates removed", "affected": exact_duplicates},
    {"rule": "365243 date placeholders converted", "affected": placeholder_total},
    {"rule": "Rare-category cells consolidated", "affected": rare_category_changes},
    {"rule": "Seller-area placeholders converted", "affected": int(seller_area_placeholder.sum())},
    {"rule": "Features removed by missingness/constant policy", "affected": len(removed_features)},
])
previous_clean.to_pickle(output_path)
feature_decisions.to_csv(audit_folder / "previous_application_feature_decisions.csv", index=False)
cleaning_audit.to_csv(audit_folder / "previous_application_cleaning_audit.csv", index=False)
validation_checks.to_csv(audit_folder / "previous_application_cleaning_validation.csv", index=False)
print("Clean previous-application dataset saved:", output_path)
print("Output rows:", len(previous_clean))
print("Output columns:", previous_clean.shape[1])
print("Unique applicants:", previous_clean["SK_ID_CURR"].nunique())
print("Remaining numerical missing values:", int(previous_clean.select_dtypes(include="number").isna().sum().sum()))

Clean previous-application dataset saved: /Users/taranveersingh/A-MRP/data/interim/previous_application_clean.pkl
Output rows: 1413701
Output columns: 36
Unique applicants: 291057
Remaining numerical missing values: 3366919


## Main cleaning results

The cleaned previous-application data contains 1,413,701 records for 291,057 applicants. All previous-application IDs are complete and unique, and no duplicate rows remain.

The 365243 placeholder was converted to missing in the date columns, and rare categories were grouped together. Unusual values, like a credit amount higher than requested or a zero requested amount, were kept and flagged instead of removed.

8 features were removed because they were missing too much data in the training set or ended up constant. The cleaned data has 36 columns. The next step is to clean the instalment-payments data.
